# pgqueuer: a job queue that is just Postgres

[PgQueuer](https://github.com/janbjorge/pgqueuer) keeps jobs as rows in an ordinary table and
builds a queue out of three Postgres features:

- **`FOR UPDATE SKIP LOCKED`**: a worker claims rows nobody else has locked, so any number of
  workers can pull from one table without handing the same job out twice;
- **`LISTEN` / `NOTIFY`**: a trigger on the table notifies a channel on every insert, so idle
  workers wake up within milliseconds instead of polling;
- **transactions**: a job enqueued in the same transaction as the data it refers to exists if and
  only if that data was committed.

This notebook takes those apart one at a time, with workers running as asyncio tasks right here in
the kernel, and then covers what the library adds on top: priorities, delayed jobs, dedupe keys,
retries with backoff, held failures, a concurrency limit enforced across workers, and cron
schedules. `app/jobs.py` uses the same pieces for the doc-search ingest (see the README).

Every job here uses an entrypoint starting with `nb_`, and only those are cleared, so a running
`just queue-worker` and its `ingest_page` jobs are left alone.

**Prerequisites**: the server named in `SERVER` is up (`just db`, or `just db pg18`) and the kernel
is this project's `.venv`. The setup cell installs pgqueuer's tables if they are missing, the same
as `just pgq install`.

In [1]:
# Parameters. Papermill overrides these with `-p NAME value`.
SERVER = "pg16"  # any server in src/settings.py: "pg16", "pg18", ...

In [2]:
import asyncio
import pathlib
import sys
import time
from datetime import timedelta

from dotenv import load_dotenv

ROOT = next(
    p
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "src" / "db.py").exists()
)
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

import asyncpg
import pandas as pd
import psycopg
from pgqueuer import PgQueuer, Queries
from pgqueuer.executors import DatabaseRetryEntrypointExecutor, EntrypointExecutorParameters
from pgqueuer.models import Job
from pgqueuer.types import QueueExecutionMode

from settings import settings

DSN = settings.sqlalchemy_url(SERVER).replace("postgresql+psycopg://", "postgresql://")
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

# One asyncpg connection for looking at the tables and for enqueueing. pgqueuer also has a psycopg
# driver; section 11 shows why this notebook (and app/jobs.py) use asyncpg for the workers.
conn = await asyncpg.connect(DSN)
queries = Queries.from_asyncpg_connection(conn)

if not await queries.schema_is_installed():
    await queries.install()
    print("installed pgqueuer's schema")


async def q(sql: str, *params) -> pd.DataFrame:
    '''Run a query on the shared connection and return the rows as a DataFrame.'''
    rows = await conn.fetch(sql, *params)
    return pd.DataFrame([dict(r) for r in rows])


async def reset() -> None:
    '''Forget every nb_ job, its log rows and statistics; other entrypoints are untouched.'''
    await conn.execute("DELETE FROM pgqueuer WHERE entrypoint LIKE 'nb\\_%'")
    await conn.execute("DELETE FROM pgqueuer_log WHERE entrypoint LIKE 'nb\\_%'")
    await conn.execute("DELETE FROM pgqueuer_statistics WHERE entrypoint LIKE 'nb\\_%'")


async def worker(register, mode=QueueExecutionMode.drain, driver="asyncpg", **run_kw) -> tuple[PgQueuer, asyncio.Task]:
    '''A worker as a task: its own connection, the entrypoints `register(pgq)` adds, then qm.run().

    Only the QueueManager runs (no scheduler), and it closes its connection when it stops.
    '''
    if driver == "asyncpg":
        wconn = await asyncpg.connect(DSN)
        pgq = PgQueuer.from_asyncpg_connection(wconn)
    else:  # psycopg's driver needs autocommit: it LISTENs on the connection
        wconn = await psycopg.AsyncConnection.connect(DSN, autocommit=True)
        pgq = PgQueuer.from_psycopg_connection(wconn)
    register(pgq)

    async def run() -> None:
        try:
            await pgq.qm.run(mode=mode, dequeue_timeout=timedelta(seconds=run_kw.pop("dequeue_s", 5)), **run_kw)
        finally:
            await wconn.close()

    return pgq, asyncio.create_task(run())


await reset()
print(SERVER, (await q("SELECT version()")).iloc[0, 0].split(" on ")[0])
await q("""
    SELECT c.relname AS object, CASE c.relkind WHEN 'r' THEN 'table' ELSE c.relkind::text END AS kind,
           c.relpersistence = 'u' AS unlogged, pg_size_pretty(pg_total_relation_size(c.oid)) AS size
    FROM pg_class c WHERE c.relname LIKE 'pgqueuer%' AND c.relkind = 'r' ORDER BY 1
""")

pg16 PostgreSQL 16.15 (Debian 16.15-1.pgdg12+2)


,object,kind,unlogged,size
0,pgqueuer,table,False,192 kB
1,pgqueuer_log,table,False,144 kB
2,pgqueuer_schedules,table,False,48 kB
3,pgqueuer_statistics,table,False,24 kB


## 1. A job is a row

`enqueue` inserts into `pgqueuer`: an entrypoint (the name a worker registers a function under),
an opaque `bytea` payload, a priority, and `execute_after`. `status` starts as `queued`. Nothing
else is needed to have a queue: a worker is anything that can turn `queued` into `picked`
without colliding with another worker.

In [3]:
ids = await queries.enqueue(
    ["nb_hello"] * 3,
    [b"alpha", b"beta", b"gamma"],
    [0, 0, 0],
)
print("job ids:", ids)
await q("""
    SELECT id, entrypoint, convert_from(payload, 'UTF8') AS payload, priority, status,
           execute_after - created AS delay, attempts, queue_manager_id
    FROM pgqueuer WHERE entrypoint LIKE 'nb\\_%' ORDER BY id
""")

job ids: [290, 291, 292]


,id,entrypoint,payload,priority,status,delay,attempts,queue_manager_id
0,290,nb_hello,alpha,0,queued,0 days,0,None
1,291,nb_hello,beta,0,queued,0 days,0,None
2,292,nb_hello,gamma,0,queued,0 days,0,None


### The trigger and the channel

An insert also fires `tg_pgqueuer_changed`, which calls `pg_notify` on the channel `ch_pgqueuer`
with a small JSON event. Listen on a second connection, enqueue one job, and read what arrives.
This is how an idle worker learns about new work without polling.

In [4]:
arrived = asyncio.get_running_loop().create_future()
listener = await asyncpg.connect(DSN)
# asyncpg runs the callback as soon as the notification arrives, no polling
await listener.add_listener("ch_pgqueuer", lambda c, pid, channel, payload: arrived.done() or arrived.set_result((time.perf_counter(), channel, payload)))
t0 = time.perf_counter()
await queries.enqueue("nb_hello", b"delta")
t_note, channel, payload = await asyncio.wait_for(arrived, 5)
await listener.close()
print(f"notification {1000 * (t_note - t0):.1f} ms after the enqueue call started, on channel {channel!r}:")
print(payload)

notification 2.6 ms after the enqueue call started, on channel 'ch_pgqueuer':
{"channel" : "ch_pgqueuer", "operation" : "insert", "sent_at" : "2026-10-16T10:03:38.272643+00:00", "table" : "pgqueuer", "type" : "table_changed_event"}


### `SKIP LOCKED`, by hand

Before letting pgqueuer do it, here is the core of every dequeue. Two transactions both ask for
"the two oldest queued rows", with `FOR UPDATE SKIP LOCKED`. The first locks two rows; the second,
instead of waiting for those locks, skips them and gets the next ones. Without `SKIP LOCKED` the
second would block until the first commits, and then both would have picked the same rows.

In [5]:
PICK = """
    SELECT id, convert_from(payload, 'UTF8') AS payload FROM pgqueuer
    WHERE entrypoint = 'nb_hello' AND status = 'queued'
    ORDER BY id LIMIT 2
    FOR UPDATE SKIP LOCKED
"""
a, b = await asyncpg.connect(DSN), await asyncpg.connect(DSN)
tx_a, tx_b = a.transaction(), b.transaction()
await tx_a.start()
await tx_b.start()
rows_a = await a.fetch(PICK)  # A's transaction stays open, holding its row locks
rows_b = await b.fetch(PICK)
print("transaction A got", [tuple(r) for r in rows_a])
print("transaction B got", [tuple(r) for r in rows_b])
await tx_a.rollback()
await tx_b.rollback()
await a.close()
await b.close()

transaction A got [(290, 'alpha'), (291, 'beta')]
transaction B got [(292, 'gamma'), (293, 'delta')]


## 2. A worker

Register a function under the entrypoint with `@pgq.entrypoint(...)` and run the queue manager.
`mode=drain` processes everything that is queued and then returns, which suits a notebook (and
batch jobs, `just queue-worker drain`); the default `continuous` mode waits for more.

`pgqueuer_log` records every state change, so a job's life is visible after the fact:
`queued` at insert, `picked` when a worker claims it, then `successful` or `exception`. When a job
finishes, its row leaves `pgqueuer`; the queue table only holds work that is still to do.

In [6]:
seen = []


def register_hello(pgq: PgQueuer) -> None:
    @pgq.entrypoint("nb_hello")
    async def hello(job: Job) -> None:
        seen.append((job.id, job.payload.decode()))


pgq, task = await worker(register_hello)
await task
print("processed:", seen)
print("left in the queue:", (await q("SELECT count(*) FROM pgqueuer WHERE entrypoint LIKE 'nb\\_%'")).iloc[0, 0])
log = await q("""
    SELECT job_id, status, created FROM pgqueuer_log
    WHERE entrypoint = 'nb_hello' ORDER BY job_id, created
""")
log.pivot_table(index="job_id", columns="status", values="created", aggfunc="first")

processed: [(290, 'alpha'), (291, 'beta'), (292, 'gamma'), (293, 'delta')]
left in the queue: 0


status,picked,queued,successful
job_id,,,
290,2026-10-16 10:03:38.344682+00:00,2026-10-16 10:03:38.235078+00:00,2026-10-16 10:03:38.361927+00:00
291,2026-10-16 10:03:38.344682+00:00,2026-10-16 10:03:38.235078+00:00,2026-10-16 10:03:38.361927+00:00
292,2026-10-16 10:03:38.344682+00:00,2026-10-16 10:03:38.235078+00:00,2026-10-16 10:03:38.361927+00:00
293,2026-10-16 10:03:38.344682+00:00,2026-10-16 10:03:38.272643+00:00,2026-10-16 10:03:38.361927+00:00


## 3. Two workers, one table

Forty jobs that each sleep 50 ms, and two workers started at the same time. Each job records the
worker (`queue_manager_id`) that ran it. With `SKIP LOCKED` in the dequeue query the two split the
work without coordinating, and no job runs twice. `batch_size=4` means each dequeue claims up to
four rows, so the split is not perfectly even.

In [7]:
await reset()
await queries.enqueue(["nb_sleep"] * 40, [str(i).encode() for i in range(40)], [0] * 40)
ran = []


def register_sleep(pgq: PgQueuer) -> None:
    @pgq.entrypoint("nb_sleep")
    async def sleep(job: Job) -> None:
        await asyncio.sleep(0.05)
        ran.append({"job": int(job.payload), "worker": str(job.queue_manager_id)[:8]})


t0 = time.perf_counter()
workers = [await worker(register_sleep, batch_size=4) for _ in range(2)]
await asyncio.gather(*(t for _, t in workers))
wall = time.perf_counter() - t0

ran_df = pd.DataFrame(ran)
print(f"{len(ran_df)} runs of {ran_df['job'].nunique()} distinct jobs in {wall:.2f}s "
      f"(one worker running them one by one would need {40 * 0.05:.1f}s)")
ran_df.groupby("worker").size().rename("jobs").to_frame()

40 runs of 40 distinct jobs in 0.10s (one worker running them one by one would need 2.0s)


,jobs
worker,
b43c85c2,28
d9b44d65,12


Each worker also ran its batch of jobs concurrently (they are coroutines in one event loop), which
is why the wall time is far below forty times 50 ms even with only two workers. For CPU-bound jobs
that concurrency is an illusion; `concurrency_limit` (section 8) and `asyncio.to_thread` are the
tools there.

## 4. Waking up: `NOTIFY` instead of polling

A continuous worker blocks for up to `dequeue_timeout` waiting for work, but a notification ends
the wait early. Start one with a 30 s timeout, let it go idle, then enqueue jobs one at a time and
measure how long each waits between `enqueue` and the start of the function. If the worker were
polling every 30 s the average would be 15 s.

In [8]:
await reset()
started: dict[int, float] = {}


def register_ping(pgq: PgQueuer) -> None:
    @pgq.entrypoint("nb_ping")
    async def ping(job: Job) -> None:
        started[int(job.payload)] = time.perf_counter()


pgq, task = await worker(register_ping, mode=QueueExecutionMode.continuous, dequeue_s=30)
await asyncio.sleep(1)  # let it reach its idle wait

sent: dict[int, float] = {}
for i in range(10):
    sent[i] = time.perf_counter()
    await queries.enqueue("nb_ping", str(i).encode())
    await asyncio.sleep(0.3)
await asyncio.sleep(0.5)
pgq.qm.shutdown.set()
await task

lat = pd.Series({i: 1000 * (started[i] - sent[i]) for i in started}, name="enqueue -> start ms")
lat.describe()[["count", "min", "50%", "max"]].round(1)

count    10.0
min       4.9
50%       6.2
max       8.4
Name: enqueue -> start ms, dtype: float64

## 5. Priority and delayed jobs

Dequeue orders by `priority DESC`, then by id, and skips rows whose `execute_after` is still in the
future. Enqueue six jobs in one call: four with different priorities and two delayed by a second.
The entrypoint has `concurrency_limit=1` and the worker `batch_size=1`, so jobs run strictly one at
a time and the order the function sees is the dequeue order. The delayed ones run last, about a second later, even though
one of them has the highest priority.

In [9]:
await reset()
order = []


def register_prio(pgq: PgQueuer) -> None:
    @pgq.entrypoint("nb_prio", concurrency_limit=1)
    async def prio(job: Job) -> None:
        order.append({"payload": job.payload.decode(), "priority": job.priority, "t": time.perf_counter()})


names = ["low", "normal", "high", "urgent", "delayed-normal", "delayed-urgent"]
prios = [-5, 0, 5, 10, 0, 10]
delays = [None, None, None, None, timedelta(seconds=1), timedelta(seconds=1)]
t0 = time.perf_counter()
await queries.enqueue(["nb_prio"] * 6, [n.encode() for n in names], prios, execute_after=delays)

pgq, task = await worker(register_prio, batch_size=1)
await task
out = pd.DataFrame(order)
out["ms after enqueue"] = (1000 * (out.pop("t") - t0)).round(0)
out

,payload,priority,ms after enqueue
0,urgent,10,64.0
1,high,5,71.0
2,normal,0,74.0
3,low,-5,77.0
4,delayed-urgent,10,1006.0
5,delayed-normal,0,1010.0


## 6. Dedupe keys

A `dedupe_key` is unique among jobs that are still queued or picked. Enqueueing a second job with
the same key raises `DuplicateJobError`, or with `on_conflict="skip"` returns `None` for it instead
of an id. This is what makes `just queue-ingest` safe to run twice: each page is keyed by its path.
Once the job has finished, the key is free again.

In [10]:
await reset()
first = await queries.enqueue("nb_page", b"orm/quickstart.rst", dedupe_key="page:orm/quickstart.rst")
again = await queries.enqueue(
    ["nb_page", "nb_page"],
    [b"orm/quickstart.rst", b"core/pooling.rst"],
    [0, 0],
    dedupe_key=["page:orm/quickstart.rst", "page:core/pooling.rst"],
    on_conflict="skip",
)
print("first enqueue:", first)
print("second batch: ", again, " (None = skipped, already queued)")

from pgqueuer.errors import DuplicateJobError

try:
    await queries.enqueue("nb_page", b"orm/quickstart.rst", dedupe_key="page:orm/quickstart.rst")
except DuplicateJobError as e:
    print("default on_conflict='raise':", type(e).__name__, e.dedupe_key)

first enqueue: [350]
second batch:  [None, 352]  (None = skipped, already queued)
default on_conflict='raise': DuplicateJobError ['page:orm/quickstart.rst']


## 7. Failures: retries, held jobs, requeue

By default an exception in a job is logged and the job is deleted. Two settings change that:

- `executor_factory=DatabaseRetryEntrypointExecutor` turns an exception into a re-queue with
  exponential backoff (`initial_delay * backoff_multiplier ** attempts`), up to `max_attempts`;
  the job goes back to `queued` with `execute_after` in the future and `attempts` incremented;
- `on_failure="hold"` keeps a job that has run out of attempts in the table with status `failed`,
  where `list_failed_jobs` finds it and `requeue_jobs` puts it back (the CLI equivalents are
  `pgq failed` and `pgq requeue`).

Two jobs: `flaky` fails twice and then succeeds, `broken` always fails.

In [11]:
await reset()
attempts_seen = []


def fast_retry(params: EntrypointExecutorParameters) -> DatabaseRetryEntrypointExecutor:
    return DatabaseRetryEntrypointExecutor(
        params, max_attempts=3, initial_delay=timedelta(milliseconds=100), backoff_multiplier=2.0
    )


def register_fail(pgq: PgQueuer) -> None:
    @pgq.entrypoint("nb_fail", on_failure="hold", executor_factory=fast_retry)
    async def fail(job: Job) -> None:
        name = job.payload.decode()
        attempts_seen.append({"job": name, "attempt": job.attempts, "t": time.perf_counter()})
        if name == "broken" or job.attempts < 2:
            raise RuntimeError(f"{name} failed on attempt {job.attempts}")


t0 = time.perf_counter()
await queries.enqueue(["nb_fail"] * 2, [b"flaky", b"broken"], [0, 0])
pgq, task = await worker(register_fail)
await task
tries = pd.DataFrame(attempts_seen)
tries["ms"] = (1000 * (tries.pop("t") - t0)).round(0)
tries

,job,attempt,ms
0,flaky,0,42.0
1,broken,0,43.0
2,flaky,1,149.0
3,broken,1,149.0
4,flaky,2,356.0
5,broken,2,356.0
6,broken,3,763.0


In [12]:
failed = await queries.list_failed_jobs()
failed = [j for j in failed if j.entrypoint == "nb_fail"]
print("held as failed:", [(j.id, j.payload.decode(), j.attempts) for j in failed])

# the traceback of the last attempt is in the log
await q("""
    SELECT job_id, status, traceback->>'exception_type' AS exception, traceback->>'exception_message' AS message
    FROM pgqueuer_log WHERE entrypoint = 'nb_fail' AND traceback IS NOT NULL ORDER BY created
""")

held as failed: [(355, 'broken', 3)]


,job_id,status,exception,message
0,354,queued,RetryRequested,flaky failed on attempt 0
1,355,queued,RetryRequested,broken failed on attempt 0
2,354,queued,RetryRequested,flaky failed on attempt 1
3,355,queued,RetryRequested,broken failed on attempt 1
4,355,queued,RetryRequested,broken failed on attempt 2
5,355,failed,RuntimeError,broken failed on attempt 3


In [13]:
# Fix the cause (here: nothing to fix, so the job will fail again), then put it back in the queue.
await queries.requeue_jobs([j.id for j in failed])
await q("SELECT id, convert_from(payload, 'UTF8') AS payload, status, attempts FROM pgqueuer WHERE entrypoint = 'nb_fail'")

,id,payload,status,attempts
0,355,broken,queued,0


## 8. A concurrency limit that holds across workers

`concurrency_limit=N` on an entrypoint caps how many of its jobs are picked at once, in total. It
is enforced inside the dequeue query (a count of picked rows for that entrypoint), so it holds no
matter how many workers run, which is what an in-process semaphore cannot do. Three workers, twelve
jobs of 200 ms each, a limit of 2: the observed peak is 2, and the wall time is about
12 / 2 x 200 ms.

In [14]:
await reset()
running = 0
peak = 0


def register_limited(pgq: PgQueuer) -> None:
    @pgq.entrypoint("nb_limited", concurrency_limit=2)
    async def limited(job: Job) -> None:
        global running, peak
        running += 1
        peak = max(peak, running)
        await asyncio.sleep(0.2)
        running -= 1


await queries.enqueue(["nb_limited"] * 12, [None] * 12, [0] * 12)
t0 = time.perf_counter()
workers = [await worker(register_limited) for _ in range(3)]
await asyncio.gather(*(t for _, t in workers))
print(f"peak concurrent jobs across 3 workers: {peak}, wall {time.perf_counter() - t0:.2f}s")

peak concurrent jobs across 3 workers: 2, wall 1.71s


## 9. Enqueue in the same transaction as the data

The reason to keep a queue in the application's database: the job can be written in the same
transaction as the row it is about. Below, a document row and its "embed this" job are written
together. When the transaction rolls back, neither exists; when it commits, both do. With a separate
broker (Redis, SQS) the same code has a window where one is written and the other is not, and needs
an outbox table to close it.

`queries` runs its `INSERT` on `conn`, the same connection the document is written on, so inside
`conn.transaction()` it is part of that transaction.

In [15]:
await reset()
await conn.execute("CREATE TABLE IF NOT EXISTS nb_documents (id serial PRIMARY KEY, body text)")


async def add_document(body: str, fail: bool) -> None:
    async with conn.transaction():
        doc_id = await conn.fetchval("INSERT INTO nb_documents (body) VALUES ($1) RETURNING id", body)
        await queries.enqueue("nb_embed", str(doc_id).encode())
        if fail:
            raise RuntimeError("something went wrong after both inserts")


for body, fail in (("rolled back", True), ("committed", False)):
    try:
        await add_document(body, fail)
    except RuntimeError as e:
        print("rolled back:", e)

print(await q("SELECT id, body FROM nb_documents"))
print(await q("SELECT id, entrypoint, convert_from(payload, 'UTF8') AS doc_id FROM pgqueuer WHERE entrypoint = 'nb_embed'"))

rolled back: something went wrong after both inserts
   id       body
0   2  committed
    id entrypoint doc_id
0  369   nb_embed      2


## 10. Schedules

`@pgq.schedule(name, cron)` registers a recurring job. Each worker that runs the scheduler upserts
the schedule into `pgqueuer_schedules`, and when it is due, the first worker to claim the row (the
same `SKIP LOCKED` idea) runs it, so a schedule fires once per period however many workers there
are. `app/jobs.py` uses one to re-queue every page at 03:00.

Cron has minute resolution, so instead of waiting for one to fire, start a scheduler for two
seconds and look at what it registered.

In [16]:
from pgqueuer.models import Schedule

sconn = await asyncpg.connect(DSN)
spgq = PgQueuer.from_asyncpg_connection(sconn)


@spgq.schedule("nb_nightly", "0 3 * * *")
async def nightly(schedule: Schedule) -> None:
    print("would re-queue everything")


@spgq.schedule("nb_every_minute", "* * * * *")
async def every_minute(schedule: Schedule) -> None:
    print("tick", schedule.next_run)


stask = asyncio.create_task(spgq.sm.run())
await asyncio.sleep(2)
spgq.sm.shutdown.set()
await stask
await sconn.close()

await q("""
    SELECT entrypoint, expression, status, next_run, last_run
    FROM pgqueuer_schedules WHERE entrypoint LIKE 'nb\\_%' ORDER BY entrypoint
""")

,entrypoint,expression,status,next_run,last_run
0,nb_every_minute,* * * * *,queued,2026-10-16 10:04:00+00:00,None
1,nb_nightly,0 3 * * *,queued,2026-10-17 03:00:00+00:00,None


## 11. Drivers: why asyncpg

pgqueuer can run on asyncpg or on psycopg 3. They do not behave the same. The psycopg driver reads
notifications with `notifies(timeout=1.0)`, which holds the connection's lock for up to a second at
a time, and every query the worker loop sends in between waits for it. A job that needs a few
round trips before the next one can start then pays seconds, not milliseconds. asyncpg delivers
notifications through a callback and never blocks the connection.

Five jobs on an entrypoint with `concurrency_limit=1`, once per driver. That setup is the worst case,
because a slot frees up only after the finished job's row is deleted, and the next pick needs
another dequeue.

In [17]:
async def five_limited(driver: str) -> dict:
    await reset()
    starts = []

    def register(pgq: PgQueuer) -> None:
        @pgq.entrypoint("nb_driver", concurrency_limit=1)
        async def job(job: Job) -> None:
            starts.append(time.perf_counter())

    await queries.enqueue(["nb_driver"] * 5, [None] * 5, [0] * 5)
    t0 = time.perf_counter()
    _, task = await worker(register, driver=driver, batch_size=1)
    await task
    return {"driver": driver, "first start s": starts[0] - t0, "last start s": starts[-1] - t0,
            "drained after s": time.perf_counter() - t0}


pd.DataFrame([await five_limited("asyncpg"), await five_limited("psycopg")]).set_index("driver").round(2)

,first start s,last start s,drained after s
driver,,,
asyncpg,0.05,0.06,0.06
psycopg,0.03,10.07,15.09


The two drivers run the same SQL, so this is about the client, not Postgres. Use asyncpg for workers.
psycopg is fine for *enqueueing* from an application that already uses it (the transactional
pattern in section 9 works the same way there, with `conn.transaction()` on an autocommit
connection), because enqueueing never waits on notifications.

## 12. Cleanup

Removes the notebook's jobs, log rows, statistics, schedules and the `nb_documents` table. The
pgqueuer schema stays installed; `just pgq uninstall` removes it.

In [18]:
await reset()
await conn.execute("DELETE FROM pgqueuer_schedules WHERE entrypoint LIKE 'nb\\_%'")
await conn.execute("DROP TABLE IF EXISTS nb_documents")
await conn.close()
print("clean")

clean
